# MAFF-Net Training on Google Colab

**GridDensityBEV + AMP (FP16) + Batch Size 4**

Paper settings: RTX 4090, batch_size=4, FP16

Colab T4/A100 ile paper sonuçlarını yakalamayı hedefliyoruz.

## 1. GPU Check

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Repo'yu Drive'dan kopyala veya git clone yap
# Option A: Drive'da repo varsa
# !cp -r /content/drive/MyDrive/MAFF-Net /content/MAFF-Net

# Option B: Git clone
!git clone https://github.com/TRV-Lab/MAFF-Net.git /content/MAFF-Net

# GridDensityBEV ve AMP degisikliklerini Drive'dan kopyala
# (Eger repo'yu local'den Drive'a yuklediysen bu adimi atla)
# !cp /content/drive/MyDrive/MAFF-Net/pcdet/models/backbones_image/CQCA_cfa.py /content/MAFF-Net/pcdet/models/backbones_image/CQCA_cfa.py
# !cp /content/drive/MyDrive/MAFF-Net/tools/train_utils/train_utils.py /content/MAFF-Net/tools/train_utils/train_utils.py
# !cp /content/drive/MyDrive/MAFF-Net/tools/train.py /content/MAFF-Net/tools/train.py
# !cp /content/drive/MyDrive/MAFF-Net/tools/cfgs/MAFF-Net/MAFF-Net_vod_colab.yaml /content/MAFF-Net/tools/cfgs/MAFF-Net/

## 3. Install Dependencies

In [ ]:
%cd /content/MAFF-Net

# PyTorch (Colab genelde yuklu, kontrol et)
!pip install spconv-cu118  # Colab CUDA versiyonuna gore ayarla
!pip install tensorboardX easydict pyyaml scikit-image tqdm einops SharedArray numba

# MAFF-Net kurulumu
!python setup.py develop

## 4. Dataset Preparation

VoD dataset'ini Google Drive'a yuklemen gerekiyor.

Yapisi:
```
Drive/MyDrive/VoD/view_of_delft_PUBLIC/radar_5frames/
    ├── ImageSets/
    ├── training/
    │   ├── calib/
    │   ├── velodyne/
    │   ├── image_2/
    │   └── label_2/
    └── testing/
```

In [ ]:
# Dataset symlink
!mkdir -p data/VoD
!ln -s /content/drive/MyDrive/VoD/view_of_delft_PUBLIC data/VoD/view_of_delft_PUBLIC

# Data infos olustur (ilk seferde gerekli)
!python -m pcdet.datasets.vod.vod_dataset_radar create_vod_infos tools/cfgs/dataset_configs/vod_dataset_radar.yaml

## 5. Training

Paper ayarlari:
- **Batch size**: 4 (gercek, accumulation degil)
- **AMP**: FP16 mixed precision
- **Epochs**: 60
- **LR**: 0.01 (OneCycle)

In [ ]:
%cd /content/MAFF-Net/tools

!python train.py \
    --cfg_file cfgs/MAFF-Net/MAFF-Net_vod_colab.yaml \
    --batch_size 4 \
    --workers 4 \
    --epochs 60 \
    --amp \
    --extra_tag colab_bs4_amp

## 6. Checkpoint'lari Drive'a Kaydet

In [ ]:
# Egitim bitince checkpoint'lari Drive'a kopyala
!cp -r /content/MAFF-Net/output/cfgs/MAFF-Net/MAFF-Net_vod_colab/colab_bs4_amp/ \
       /content/drive/MyDrive/MAFF-Net_results/colab_bs4_amp/

## 7. Evaluation

In [ ]:
# En iyi epoch'u evaluate et
%cd /content/MAFF-Net/tools

!python test.py \
    --cfg_file cfgs/MAFF-Net/MAFF-Net_vod_colab.yaml \
    --batch_size 4 \
    --workers 4 \
    --extra_tag colab_bs4_amp \
    --eval_all

## 8. Results Comparison

| Setting | Car | Ped | Cyc | mAP3d | mAPbev |
|---------|-----|-----|-----|-------|--------|
| Paper (RTX4090, bs=4, FP16) | 42.33 | 46.75 | 74.72 | **54.59** | **58.37** |
| Local (RTX4060, bs=1, FP32) | 25.25 | 37.26 | 66.49 | 43.00 | 46.54 |
| **Colab (T4/A100, bs=4, FP16)** | ? | ? | ? | **?** | **?** |